# 03 심화: Ablation, score 민감도, leakage와 반복 학습

표의 숫자를 그대로 외우지 않고 어떤 목표와 데이터 처리 선택이 결론을 바꾸는지 검증한다.

In [1]:
# test split ablation: (name, CS, CR, PC)
ablations = [
    ('none', 9.31, 13.26, 149.28),
    ('constraint only', 7.02, 16.44, 36.57),
    ('iterative only', 6.29, 24.67, 149.26),
    ('both', 6.28, 19.25, 40.91),
]
print('최저 CS:', min(ablations, key=lambda x:x[1]))
print('최고 CR:', max(ablations, key=lambda x:x[2]))
print('최저 PC:', min(ablations, key=lambda x:x[3]))

최저 CS: ('both', 6.28, 19.25, 40.91)
최고 CR: ('iterative only', 6.29, 24.67, 149.26)
최저 PC: ('constraint only', 7.02, 16.44, 36.57)


최종 모델은 CR 최고가 아니다. CS의 목적함수가 전력에 충분한 가중치를 주기 때문에 constraint와 iterative training을 모두 쓴 모델이 선택된다.

In [2]:
# 전력 가중치가 달라질 때 단순 proxy ranking이 어떻게 바뀌는지 본다.
def proxy_score(cr, pc, power_weight):
    return 100/(cr+1e-9) + power_weight*pc

for weight in (0, 0.005, 0.01, 0.03, 0.1):
    ranked = sorted((round(proxy_score(cr,pc,weight),3), name)
                    for name,_,cr,pc in ablations)
    print(f'power_weight={weight:0.3f}:', ranked)

power_weight=0.000: [(4.054, 'iterative only'), (5.195, 'both'), (6.083, 'constraint only'), (7.541, 'none')]
power_weight=0.005: [(4.8, 'iterative only'), (5.399, 'both'), (6.266, 'constraint only'), (8.288, 'none')]
power_weight=0.010: [(5.546, 'iterative only'), (5.604, 'both'), (6.448, 'constraint only'), (9.034, 'none')]
power_weight=0.030: [(6.422, 'both'), (7.18, 'constraint only'), (8.531, 'iterative only'), (12.02, 'none')]
power_weight=0.100: [(9.286, 'both'), (9.74, 'constraint only'), (18.98, 'iterative only'), (22.469, 'none')]


## Normalization leakage 실험

test sample까지 포함한 평균·표준편차로 normalize하면 test distribution 정보를 미리 사용한다. 아래는 train-only와 전체 데이터 통계가 동일한 test 값을 다르게 바꾸는 예다.

In [3]:
from statistics import mean, pstdev
train = [1.0, 1.2, 0.8, 1.1, 0.9]
test = [4.0, 4.2]
def zscore(values, reference):
    mu, sd = mean(reference), pstdev(reference)
    return [(x-mu)/sd for x in values]

train_only = zscore(test, train)
leaky = zscore(test, train+test)
print('train-only:', [round(x,2) for x in train_only])
print('all-split(leaky):', [round(x,2) for x in leaky])
assert max(leaky) < max(train_only)

train-only: [21.21, 22.63]
all-split(leaky): [1.5, 1.65]


## Simulation-based iterative filtering

모델 proposal 가운데 임계값을 넘은 것만 다시 학습 집합에 넣는다. 이 과정은 성능을 높일 수 있지만, 다양성이 줄고 현재 score의 편향을 강화할 수도 있다.

In [4]:
import random
rng = random.Random(7)
dataset_scores = [rng.gauss(0.55,0.08) for _ in range(100)]
history = []
for cycle in range(4):
    proposal = [rng.gauss(mean(dataset_scores)+0.02,0.10) for _ in range(80)]
    threshold = 0.62
    accepted = [min(1,max(0,x)) for x in proposal if x > threshold]
    dataset_scores.extend(accepted)
    history.append((cycle, len(accepted), mean(dataset_scores), pstdev(dataset_scores)))
history

[(0, 28, 0.573048357068142, 0.08895970307236785),
 (1, 36, 0.6020291196959321, 0.09829365489630669),
 (2, 41, 0.6227007780113016, 0.10056504783954712),
 (3, 55, 0.6410292003169134, 0.10128426812569455)]

In [5]:
# 임계값별 수락률을 보고 너무 높은 threshold가 exploration을 고갈시키는지 확인한다.
proposal = [rng.random() for _ in range(1000)]
rates = {t:sum(x>t for x in proposal)/len(proposal) for t in (0.5,0.7,0.9,0.98)}
assert rates[0.5] > rates[0.7] > rates[0.9] > rates[0.98]
rates

{0.5: 0.477, 0.7: 0.296, 0.9: 0.101, 0.98: 0.016}

## 재현 체크리스트

- train statistics로만 normalization했는가?
- 여러 random seed의 평균·표준편차 또는 confidence interval을 보고했는가?
- 90k, 100k, 200k iteration 중 논문 표에 대응하는 checkpoint를 확인했는가?
- baseline마다 같은 simulator calls와 tuning budget을 부여했는가?
- sensor power뿐 아니라 자세 제어·전체 bus 전력을 포함했는가?